In [1]:
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from typing import TypedDict, Annotated, Literal
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage, SystemMessage, BaseMessage
import os
from langgraph.graph.message import add_messages
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool
import sqlite3

import random
import requests 

load_dotenv()

C:\Users\hashr\AppData\Local\Temp\ipykernel_28992\4138887897.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


True

In [2]:
import sys
print(sys.executable)

from langgraph.checkpoint.sqlite import SqliteSaver

print("SQLite import successful!")

c:\Langgraphs\AgenticAI-using-Langgraphs\venv\Scripts\python.exe
SQLite import successful!


In [3]:
load_dotenv()

True

In [4]:
api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError("GROQ_API_KEY is missing from .env")

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    groq_api_key=api_key
)

In [5]:
search_tool = DuckDuckGoSearchRun(region='us-en')

@tool
def calculator(first_num:float, second_num:float,operator:str)->dict:
    """
    Perform the basic math operations on two numbers.
    Supporting operators are add, sub, mul, div
    """
    try:
        if operator=='add':
            result = first_num+second_num
        elif operator=='sub':
            result = first_num-second_num
        elif operator=='mul':
            result = first_num*second_num
        elif operator=='div':
            if second_num==0:
                return {'error': "Division by zero is not allowed"}
            result = first_num/second_num
        else:
            return {'error':'Unsupported operator not allowed'}
        return {'first num': first_num, 'second num':second_num, 'operator':operator, 'result':result}
    except Exception as e:
        return {'error': str(e)}

# @tool
# def get_stock_price(symbol:str)->dict:
#     """
#     Fetch and give me the latest price of the stock ie TSLA etc and i am using an APi Alpha Vintage with api key in the url
#     """
#     url=f""
#     r=requests.get(url)
#     return r.json()



In [6]:
tools = [search_tool,calculator]
llm_with_tools  = llm.bind_tools(tools)

In [7]:
class chatState(TypedDict):
    messages: Annotated[list[BaseMessage],add_messages]

In [8]:
def ChatBot(state: chatState):
    messages = state["messages"]

    response = llm.invoke(messages)

    return {"messages": [response]}

In [9]:
tool_node = ToolNode(tools)
conn = sqlite3.connect(database='chatbot.db',check_same_thread=False)

In [10]:
checkpointer = SqliteSaver(conn=conn)
graph = StateGraph(chatState)
graph.add_node('ChatBot',ChatBot)
graph.add_node('tools',tool_node)

graph.add_edge(START,'ChatBot')
graph.add_conditional_edges('ChatBot',tools_condition)
graph.add_edge('tools','ChatBot')


In [11]:
chatbot = graph.compile(checkpointer=checkpointer)
chatbot

def retrieveallthreads():
    all_threads=set()
    for checkpoint in checkpointer.list(None):
        all_threads.add(checkpoint.config['configurable']['thread_id'])  
    return list(all_threads)


In [12]:
# thread_id = '1'



# while True:
#     userMessage = input('Type here:' )

#     print('user:', userMessage)

#     if userMessage.strip().lower() in  ['exit','bye','stop']:
#         break

#     config  = {'configurable': {'thread_id': thread_id}}

#     response = workflow.invoke({'messages':[HumanMessage(content=userMessage)]}, config=config)
# # 
#     print("AI:", response["messages"][-1].content)    
